# RNN Optuna Hyperparameter Search

**Что делает ноутбук:**
1. Загружает normalized `.npz` из Google Drive
2. Optuna подбирает лучшие гиперпараметры GRU (200+ trials)
3. Сохраняет `best_params.json` для финального обучения

**Output:**
- `best_params.json` → используется в `rnn_final_train_from_optuna.ipynb`
- `optuna_*/study.db` → сохраняется для возобновления поиска

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# @title 0. Check / Install dependencies
import importlib.util
import subprocess
import sys

required = {
    'optuna': 'optuna',
    'onnx': 'onnx',
    'onnxruntime': 'onnxruntime',
    'onnxscript': 'onnxscript',
}
missing = [package for module, package in required.items()
           if importlib.util.find_spec(module) is None]

if missing:
    print('Устанавливаю отсутствующие пакеты:', ', '.join(missing))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
else:
    print('Все необходимые пакеты уже установлены.')

import onnx
import onnxruntime
import onnxscript
import optuna

print(f'onnx:       {onnx.__version__}')
print(f'onnxruntime: {onnxruntime.__version__}')
print(f'onnxscript:  {getattr(onnxscript, "__version__", "installed")}')
print(f'optuna:      {optuna.__version__}')
print('Зависимости готовы.')

Все необходимые пакеты уже установлены.
onnx:       1.22.0
onnxruntime: 1.30.0
onnxscript:  0.7.2
optuna:      5.0.0
Зависимости готовы.


In [3]:
# @title 1. Imports
import os, sys, glob, math, warnings, io, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler

warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Device: cuda


In [4]:
# @title 2. Config & Paths

RNN_DIR = Path('/content/drive/MyDrive/RNN')
ARTIFACTS_DIR = RNN_DIR / 'data_normalized'
OUTPUT_DIR = RNN_DIR / 'output'

# Data contract
N_FEATURES   = 12
N_BARS       = 20
N_CLASSES    = 2
BATCH_SIZE   = 64

# Optuna
MIN_TRADE_FRACTION = 0.10
TEST_MONTHS = 5
N_TRIALS = 10
TRAIN_EPOCHS = 40

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Artifacts: {ARTIFACTS_DIR}')
print(f'Output:    {OUTPUT_DIR}')
print(f'Optuna trials: {N_TRIALS}')

Artifacts: /content/drive/MyDrive/RNN/data_normalized
Output:    /content/drive/MyDrive/RNN/output
Optuna trials: 10


In [5]:
# @title 3. Data Loader

def load_and_split_npz(filepath, seq_len, eps=1e-6):
    data = np.load(filepath, allow_pickle=True)
    X = data['X'].astype(np.float32)
    y = data['y'].astype(int)
    datetimes = pd.to_datetime(data['datetime'])

    n_total = len(X)
    actual_seq = min(seq_len, N_BARS)
    X = X[:, :actual_seq, :]

    test_start_date = datetimes.max() - pd.DateOffset(months=TEST_MONTHS)
    n_train = int(np.searchsorted(datetimes.values, test_start_date.to_datetime64(), side='left'))
    n_train = min(max(n_train, 1), n_total - 1)

    X_train, X_test = X[:n_train], X[n_train:]
    y_train, y_test = y[:n_train], y[n_train:]

    num_classes = max(2, len(np.unique(y)))
    y_train_oh = np.eye(num_classes)[y_train]
    y_test_oh  = np.eye(num_classes)[y_test]

    print(f"  rows={n_total}, train={n_train}, test={n_total-n_train}, X={X.shape}")

    train_ds = TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train_oh))
    test_ds  = TensorDataset(torch.FloatTensor(X_test),  torch.FloatTensor(y_test_oh))
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

    return train_loader, test_loader, y_test, actual_seq

buy_npz = sorted(glob.glob(str(ARTIFACTS_DIR / 'data_BUY_*_normalized.npz')))
if not buy_npz:
    raise FileNotFoundError(f'BUY NPZ missing: {ARTIFACTS_DIR}/')
print(f"BUY files: {[Path(f).name for f in buy_npz]}")

BUY files: ['data_BUY_20240102_20260629_normalized.npz']


In [6]:
# @title 4. GRU Model

class GRUClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes, dropout=0.2):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True,
                          dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size, device=x.device)
        out, _ = self.gru(x, h0)
        out = out[:, -1, :]
        out = self.dropout(out)
        return self.fc(out)

In [7]:
# @title 5. Classification Metrics

def classification_metrics(model, test_loader, y_test, threshold, device):
    model.eval()
    all_probs = []
    with torch.no_grad():
        for X_batch, _ in test_loader:
            logits = model(X_batch.to(device))
            probs = torch.softmax(logits, dim=1)[:, 1]
            all_probs.extend(probs.cpu().numpy())

    predicted_class = np.asarray(all_probs) > threshold
    actual_labels = y_test.astype(bool)

    tp = int(np.sum(predicted_class & actual_labels))
    fp = int(np.sum(predicted_class & ~actual_labels))
    n_trades = tp + fp
    trade_fraction = n_trades / max(len(y_test), 1)
    precision = tp / n_trades if n_trades else 0.0
    valid = trade_fraction > MIN_TRADE_FRACTION

    return {
        'tp': tp, 'fp': fp, 'n_trades': n_trades,
        'trade_fraction': trade_fraction, 'precision': precision, 'valid': valid,
    }

In [8]:
# @title 6. Training

def train_one_model(model, train_loader, epochs, lr, device, verbose=False):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr*0.1)
    best_acc, best_state = 0.0, None

    for epoch in range(epochs):
        model.train()
        loss_t, corr, tot = 0.0, 0, 0
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            yp = model(Xb)
            loss = criterion(yp, yb.argmax(dim=1))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            loss_t += loss.item() * Xb.size(0)
            corr += (yp.argmax(1) == yb.argmax(1)).sum().item()
            tot += Xb.size(0)
        scheduler.step()
        acc = corr / max(tot, 1)
        if acc > best_acc:
            best_acc = acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state:
        model.load_state_dict(best_state)
    return model, best_acc

In [9]:
# @title 7. Optuna Hyperparameter Search

def load_or_create_study(mode_name):
    study_name = f'rnn_{mode_name}_optuna'
    optuna_dir = OUTPUT_DIR / f'optuna_{mode_name}'
    optuna_dir.mkdir(exist_ok=True)
    storage_path = optuna_dir / 'study.db'
    storage = f'sqlite:///{storage_path}'

    if storage_path.exists():
        print(f'Loading study: {study_name}')
        study = optuna.load_study(study_name=study_name, storage=storage)
        done = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
        print(f'  Completed: {done} trials')
        return study

    print(f'Creating study: {study_name}')
    return optuna.create_study(
        study_name=study_name,
        storage=storage,
        load_if_exists=True,
        direction='maximize',
        sampler=TPESampler(seed=42),
        pruner=MedianPruner(n_startup_trials=10),
    )

def run_optuna(npz_path, mode_name, n_trials):
    study = load_or_create_study(mode_name)
    done = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
    remaining = n_trials - done

    if remaining > 0:
        print(f'Running {remaining} more trials...\n')

        def objective(trial):
            seq_len = trial.suggest_int('seq_len', 10, 20, step=2)
            hidden_size = trial.suggest_int('hidden_size', 32, 128, step=16)
            num_layers = trial.suggest_int('num_layers', 1, 3)
            dropout = trial.suggest_float('dropout', 0.1, 0.4, step=0.05)
            lr = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
            conf_thresh = trial.suggest_float('conf_threshold', 0.5, 0.9, step=0.1)

            train_loader, test_loader, y_test, seq_len_actual = load_and_split_npz(npz_path, seq_len)
            model = GRUClassifier(N_FEATURES, hidden_size, num_layers, N_CLASSES, dropout)
            model, _ = train_one_model(model, train_loader, TRAIN_EPOCHS, lr, DEVICE, verbose=False)
            metrics = classification_metrics(model, test_loader, y_test, conf_thresh, DEVICE)

            trial.set_user_attr('tp', metrics['tp'])
            trial.set_user_attr('fp', metrics['fp'])
            trial.set_user_attr('n_trades', metrics['n_trades'])
            trial.set_user_attr('trade_fraction', metrics['trade_fraction'])
            trial.set_user_attr('precision', metrics['precision'])

            return metrics['precision'] if metrics['valid'] else -999.0

        study.optimize(objective, n_trials=remaining, show_progress_bar=True)

    # Summary
    print(f"\n{'='*70}")
    print(f'Best trial: {study.best_trial.number}')
    print(f'Best precision: {study.best_value:.4f}')
    print(f'Best params:')
    for k, v in study.best_params.items():
        print(f'  {k}: {v}')
    print(f"{'='*70}\n")

    return study.best_params, study

buy_params, buy_study = run_optuna(buy_npz[0], 'BUY', N_TRIALS)

Loading study: rnn_BUY_optuna
  Completed: 1 trials
Running 9 more trials...



  0%|          | 0/9 [00:00<?, ?it/s]

  rows=4156, train=3705, test=451, X=(4156, 12, 12)
[I 2026-09-14 11:12:43,102] Trial 2 finished with value: -999.0 and parameters: {'seq_len': 12, 'hidden_size': 32, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.0005009280495421835, 'conf_threshold': 0.5}. Best is trial 0 with value: -999.0.
  rows=4156, train=3705, test=451, X=(4156, 12, 12)
[I 2026-09-14 11:12:51,282] Trial 3 finished with value: -999.0 and parameters: {'seq_len': 12, 'hidden_size': 80, 'num_layers': 2, 'dropout': 0.25, 'lr': 0.0005884170901610781, 'conf_threshold': 0.7}. Best is trial 0 with value: -999.0.
  rows=4156, train=3705, test=451, X=(4156, 10, 12)
[I 2026-09-14 11:12:57,713] Trial 4 finished with value: -999.0 and parameters: {'seq_len': 10, 'hidden_size': 96, 'num_layers': 1, 'dropout': 0.4, 'lr': 0.00015538763920316847, 'conf_threshold': 0.5}. Best is trial 0 with value: -999.0.
  rows=4156, train=3705, test=451, X=(4156, 20, 12)
[I 2026-09-14 11:13:04,882] Trial 5 finished with value: -999.0

In [10]:
# @title 8. Save Best Parameters for Final Training

best_trial = buy_study.best_trial
summary_optuna = {
    'best_params': {k: (float(v) if isinstance(v, (np.floating,)) else int(v) if isinstance(v, (np.integer,)) else v)
                   for k, v in buy_params.items()},
    'best_value': float(buy_study.best_value),
    'best_trial_number': best_trial.number,
    'metrics': {
        'tp': int(best_trial.user_attrs.get('tp', 0)),
        'fp': int(best_trial.user_attrs.get('fp', 0)),
        'n_trades': int(best_trial.user_attrs.get('n_trades', 0)),
        'trade_fraction': float(best_trial.user_attrs.get('trade_fraction', 0.0)),
        'precision': float(best_trial.user_attrs.get('precision', 0.0)),
    }
}

with open(OUTPUT_DIR / 'best_params.json', 'w') as f:
    json.dump(summary_optuna, f, indent=2)

print(f"\nOptuna Search Complete!")
print(f"Best params saved to: {OUTPUT_DIR}/best_params.json")
print(f"\nNext step: Run rnn_final_train_from_optuna.ipynb to train final model.")


Optuna Search Complete!
Best params saved to: /content/drive/MyDrive/RNN/output/best_params.json

Next step: Run rnn_final_train_from_optuna.ipynb to train final model.
